# AIMO3 Writeup — Kaggle-Agnostic Reproduction (Google Colab)

**Companion notebook for the AIMO3 Writeup Prize submission:** *A Practitioner's Plateau: Sixteen Falsified Modifications to the AIMO3 Public-Consensus Pipeline on gpt-oss-120b-MXFP4* by **Juan Sebastian Gil Pinzon** (Kaggle: `sebastiangil00`, team *Hail Mary*).

- Kaggle Writeup: (link in the companion Discussion comment)
- Kaggle Submission (42/50 public = 42/50 private): https://www.kaggle.com/code/sebastiangil00/aimo3-v7-winner-fork
- GitHub companion: https://github.com/SebastianGilPinzon/aimo3-writeup-public
- Kaggle Dataset: https://www.kaggle.com/datasets/sebastiangil00/aimo3-writeup-artifacts

## What this notebook does

This is a **Kaggle-agnostic** version of the competition submission: it pulls `gpt-oss-120b` directly from HuggingFace and runs the full inference pipeline (Harmony prompt + K=8 parallel rollouts + TIR via Jupyter kernel + entropy-weighted majority voting + early-stop). It produces answers for the 10 AIMO3 reference problems, matching the distribution documented in `reproducibility/expected_hashes.json`.

## Runtime requirements

- **Hardware:** single NVIDIA H100 80GB (or A100 80GB as a fallback, with reduced context). Colab Pro+ is required for H100.
- **Wall-clock:** ~25 minutes for the 10-problem reference set after model load. Model load itself is ~10–20 minutes from a cold cache (downloading ~60 GB from HuggingFace).
- **RAM:** ≥ 40 GB system RAM recommended (for `preload_model_weights`-style warm-up).

## Cells below

1. Install dependencies (vLLM 0.11.2, openai_harmony, jupyter_client).
2. Download `gpt-oss-120b` from HuggingFace (`danielhanchen/gpt-oss-120b`).
3. Pull the rest of the harness from our GitHub repo.
4. Launch vLLM with the exact same flags as the Kaggle submission.
5. Solve the 10 reference problems.
6. Write `submission.parquet` and compare against expected-answer distributions.
7. (Optional) regenerate all 4 figures from the writeup.

Reviewers who want to verify reproducibility without burning compute can skip the GPU-heavy cells and run the CPU-only trap detection tests from the GitHub repo: `pytest tests/` (15 passed in <1 s).

---

## 1. Verify GPU and install dependencies

In [ ]:
!nvidia-smi -L
# Expected: one of {NVIDIA H100 80GB HBM3, NVIDIA A100 80GB} for this notebook.
# Other GPUs will fail to fit gpt-oss-120b-MXFP4 with context 65536.

In [ ]:
# Install pinned versions matching the Kaggle submission environment
%pip install --quiet vllm==0.11.2 openai-harmony jupyter-client ipykernel sympy mpmath numpy pandas pyarrow
%pip install --quiet huggingface-hub

# Optional: test dependencies for the CPU-runnable substrate-trap detectors
%pip install --quiet pytest

## 2. Download gpt-oss-120b from HuggingFace

The base model is `danielhanchen/gpt-oss-120b` — same weights as the Kaggle submission, just mirrored outside Kaggle. Download is ~60 GB; plan for 10–20 minutes on typical Colab bandwidth.

In [ ]:
import os
from huggingface_hub import snapshot_download

MODEL_REPO = 'danielhanchen/gpt-oss-120b'
MODEL_LOCAL = '/content/gpt-oss-120b'

if not os.path.isdir(MODEL_LOCAL) or not os.listdir(MODEL_LOCAL):
    snapshot_download(
        repo_id=MODEL_REPO,
        local_dir=MODEL_LOCAL,
        local_dir_use_symlinks=False,
        max_workers=8,
    )

print('Model ready at', MODEL_LOCAL)
!ls -sh "$MODEL_LOCAL" | head -10

## 3. Pull the harness from the companion GitHub repo

The full pipeline source (identical to the Kaggle submission) lives in
`submission/notebook.py` of the companion repo. Here we clone the repo and
use its helper modules inline.

In [ ]:
!git clone --depth 1 https://github.com/SebastianGilPinzon/aimo3-writeup-public /content/aimo3-writeup
import sys
sys.path.insert(0, '/content/aimo3-writeup')
sys.path.insert(0, '/content/aimo3-writeup/submission')

# The harness notebook exports a `predict(problem_id, problem)` function that returns an int answer.
# We import it here, but deferred — the import spawns the vLLM server.
print('Repository cloned. The submission source is at:')
!wc -l /content/aimo3-writeup/submission/notebook.py

## 4. Launch vLLM server

Same command as the Kaggle submission (§3.2 of the writeup): MXFP4 quantization, fp8 KV cache, 65536 max context, `--trust-remote-code` for the gpt-oss architecture.

On Colab H100 we keep `gpu_memory_utilization=0.90` (instead of `0.94` on Kaggle) to leave headroom for vLLM's sampler warm-up.

In [ ]:
import subprocess, time, sys, os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

cmd = [
    sys.executable, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', '/content/gpt-oss-120b',
    '--served-model-name', 'gpt-oss',
    '--dtype', 'auto',
    '--quantization', 'mxfp4',
    '--kv-cache-dtype', 'fp8_e4m3',
    '--max-model-len', '65536',
    '--gpu-memory-utilization', '0.90',  # Colab headroom
    '--trust-remote-code',
    '--tensor-parallel-size', '1',
    '--host', '127.0.0.1',
    '--port', '8000',
    '--disable-log-stats',
]

log = open('/content/vllm.log', 'w')
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
print('vLLM server starting (PID', proc.pid, ')... waiting ~90 s for Triton compile.')

In [ ]:
import time, requests
for _ in range(60):
    try:
        r = requests.get('http://127.0.0.1:8000/v1/models', timeout=2)
        if r.ok:
            print('vLLM ready:', r.json()['data'][0]['id'])
            break
    except Exception:
        pass
    time.sleep(5)
else:
    raise RuntimeError('vLLM did not come up in 5 minutes; check /content/vllm.log')

## 5. Solve the 10 reference problems

We use the `local_gateway.py` harness from the companion repo in STOCHASTIC mode (matches the actual Kaggle submission; uses time-based seeds). Mode A (STRICT) is available via `REPRODUCE_DETERMINISTIC=1` for bitwise reproducibility.

Expected per-problem outputs are documented in `reproducibility/expected_hashes.json` — the reference-set distributions from 30 runs on Kaggle H100.

In [ ]:
%cd /content/aimo3-writeup
# Run the local gateway against the 10-problem reference set.
# Output: submission.parquet with {id, answer} columns.
!python reproducibility/local_gateway.py \
    --problems data/reference.csv \
    --notebook submission/notebook.py \
    --mode stochastic \
    --seed 42 \
    --output /content/submission.parquet

## 6. Verify against expected-answer distributions

In [ ]:
!python /content/aimo3-writeup/reproducibility/verify.py \
    /content/submission.parquet \
    --mode stochastic \
    --expected-json /content/aimo3-writeup/reproducibility/expected_hashes.json

## 7. Regenerate all 4 figures from the writeup (optional)

Per the host's requirement that writeup figures be reproducible from code, all 4 figures are regenerated by this single script.

Output PNGs appear at `/content/aimo3-writeup/figures/fig*.png` and match the versions embedded in the writeup.

In [ ]:
%pip install --quiet matplotlib
!cd /content/aimo3-writeup/figures && python generate_figures.py
!ls -la /content/aimo3-writeup/figures/*.png

## 8. (CPU-only, no GPU needed) substrate-trap detection tests

The three named substrate traps from §6 of the writeup (LoRA+MXFP4 collapse, EAGLE-3+MoE zero tokens, sqrt-prior Bayesian inversion) each have a runnable pytest-based detector. Two of three run on CPU; the EAGLE-3 integration test is gated on `ENABLE_H100_INTEGRATION=1`.

Expected: 15 passed, 2 skipped.

In [ ]:
!cd /content/aimo3-writeup && python -m pytest tests/ -v

---

## Differences from the Kaggle submission (§ EDIT 3 of host post 689703)

Per the host's guidance allowing "unrestricted" variants in writeup-companion notebooks, the ONLY intentional differences from our Kaggle submission (`AIMO3 v7 - Winner Fork - Version 12`) are:

| Field | Kaggle value | Colab value | Reason |
|---|---|---|---|
| `gpu_memory_utilization` | 0.94 | 0.90 | Colab H100 sampler warm-up headroom |
| `PYTORCH_ALLOC_CONF` | default | `expandable_segments:True` | Colab OOM prevention during model swizzle |

All other parameters — `temperature=1.0`, `min_p=0.02`, `top_logprobs=5`, `K=8`, `early_stop=4`, entropy-weighted voting floor `0.3`, Jupyter-kernel TIR, Harmony prompt — are byte-identical to the Kaggle submission.

The Kaggle-restricted version is the one above (as-submitted). An **unrestricted** variant would raise `max_model_len` to 131072, remove `kv_cache_dtype` quantization, and set `K=16`; we did not run this variant on Colab for cost reasons, but it is a one-argument change for any reviewer who wishes to benchmark the gap.

## Contact

Issues: https://github.com/SebastianGilPinzon/aimo3-writeup-public/issues

Kaggle handle: `sebastiangil00`